# LeetCode 451 — Sort Characters By Frequency bằng Python

Link đề: https://leetcode.com/problems/sort-characters-by-frequency/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Sorting, Heap (Priority Queue), Bucket Sort, Counting

Skeleton LeetCode:

```python
class Solution:
    def frequencySort(self, s: str) -> str:
        ...
```

Ghi chú: đề được viết lại theo hiểu biết về đề gốc; nên đối chiếu lại đề gốc nếu muốn khớp từng chữ. Bài này có **nhiều đáp án đúng** (các ký tự cùng tần suất có thể đứng theo thứ tự bất kỳ), nên notebook **kiểm tra bằng hàm kiểm tra tính hợp lệ** (`is_valid_freq_sort`), không so với đáp án cứng.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: sắp xếp lại các ký tự của chuỗi `s` theo **tần suất giảm dần**; các ký tự giống nhau phải đứng **liền nhau**; nhiều đáp án đúng thì trả về **một** đáp án bất kỳ.
2. Nắm các công cụ Python: `collections.Counter`, `dict.items()`, `sorted(..., key=...)` với `lambda` và khoá âm, nhân chuỗi `c * k`, `"".join`, hàng đợi ưu tiên `heapq` (min-heap, dùng số âm để thành max-heap), danh sách các "xô" (bucket), `itertools.groupby`.
3. Đi từ **chọn lần lượt ký tự nhiều nhất còn lại (vét)** → **đếm + sắp xếp các ký tự phân biệt** → **dùng heap** → **bucket sort theo tần suất (`O(n)`)** → **sắp xếp cả chuỗi với khoá kép**.
4. Hiểu mấu chốt: bài chia làm **hai bước**: (a) **đếm** mỗi ký tự; (b) **xếp theo số đếm giảm dần** rồi **in mỗi ký tự lặp đúng số lần của nó**. Tần suất nằm trong khoảng `1..n` nên có thể xếp bằng bucket (không cần so sánh).
5. Tránh các lỗi kinh điển: chỉ sắp theo tần suất mà không gom ký tự giống nhau (ra `"abab"` thay vì `"aabb"`), nhầm chữ hoa/chữ thường, trả về `list` thay vì `str`, xếp tăng dần thay vì giảm dần, và dùng `[[]] * k` làm bucket (các xô dùng chung một list).

## 2. Đề bài & ví dụ

Cho một chuỗi `s`. Hãy sắp xếp các ký tự của nó theo **thứ tự giảm dần của tần suất** (số lần xuất hiện). Trả về chuỗi đã sắp xếp. Nếu có nhiều đáp án, trả về **một** đáp án bất kỳ.

**Ví dụ 1**

- Input: `s = "tree"`
- Output: `"eert"`
- Giải thích: `'e'` xuất hiện 2 lần, `'r'` và `'t'` đều 1 lần. Vậy `'e'` phải đứng trước `'r'` và `'t'`. Do đó `"eetr"` cũng là đáp án hợp lệ.

**Ví dụ 2**

- Input: `s = "cccaaa"`
- Output: `"aaaccc"`
- Giải thích: `'c'` và `'a'` đều xuất hiện 3 lần nên cả `"cccaaa"` và `"aaaccc"` đều hợp lệ. Nhưng `"cacaca"` là **sai**, vì các ký tự giống nhau phải đứng **liền nhau**.

**Ví dụ 3**

- Input: `s = "Aabb"`
- Output: `"bbAa"`
- Giải thích: `"bbaA"` cũng hợp lệ nhưng `"Aabb"` thì sai. Chữ `'A'` và `'a'` được coi là **hai ký tự khác nhau**.

**Ràng buộc:**

- `1 <= s.length <= 5 * 10^5`
- `s` gồm chữ cái hoa, chữ cái thường và chữ số (tối đa `62` ký tự khác nhau).

Chú ý: do có nhiều đáp án đúng nên ta dùng **hàm kiểm tra tính hợp lệ** để chấm: (1) cùng tập ký tự (cùng số lần); (2) mỗi ký tự **liền một khối**; (3) độ dài các khối **không tăng**.

In [1]:
from collections import Counter, defaultdict
from itertools import groupby, permutations
import heapq
import random
import string
import time


def is_valid_freq_sort(s, out):
    """Kiểm tra `out` có phải một đáp án hợp lệ của bài toán với đầu vào `s` hay không."""
    if not isinstance(out, str) or Counter(out) != Counter(s):
        return False                      # (1) cùng ký tự, cùng số lần
    seen, lengths = set(), []
    for ch, grp in groupby(out):          # groupby gom các ký tự GIỐNG NHAU LIỀN KỀ thành một khối
        if ch in seen:
            return False                  # (2) ký tự này đã có một khối trước đó -> bị tách rời
        seen.add(ch)
        lengths.append(len(list(grp)))
    return all(a >= b for a, b in zip(lengths, lengths[1:]))   # (3) khối sau không dài hơn khối trước


examples = [("tree", "eert"), ("cccaaa", "aaaccc"), ("Aabb", "bbAa")]
for s, ans in examples:
    assert is_valid_freq_sort(s, ans)
    print(f"{s!r:10} -> {ans!r} hợp lệ")

# các đáp án phụ trong đề: hợp lệ / không hợp lệ
assert is_valid_freq_sort("tree", "eetr") and is_valid_freq_sort("cccaaa", "cccaaa") and is_valid_freq_sort("Aabb", "bbaA")
assert not is_valid_freq_sort("cccaaa", "cacaca") and not is_valid_freq_sort("Aabb", "Aabb")
print("Các đáp án phụ của đề cũng khớp bộ kiểm tra.")

'tree'     -> 'eert' hợp lệ
'cccaaa'   -> 'aaaccc' hợp lệ
'Aabb'     -> 'bbAa' hợp lệ
Các đáp án phụ của đề cũng khớp bộ kiểm tra.


## 3. Kiến thức nền

### 3.1. `Counter` — đếm số lần xuất hiện

`Counter(s)` nhận một chuỗi (hoặc list...) và trả về một "từ điển" `{ký tự: số lần}`. Tra ký tự **chưa có** cho `0` (không báo lỗi). Chuỗi chỉ có **tối đa 62 ký tự khác nhau** (26 + 26 + 10) nên `Counter` luôn nhỏ.

In [2]:
cnt = Counter("tree")
print(cnt)                    # Counter({'e': 2, 't': 1, 'r': 1})
print(cnt["e"], cnt["z"])     # 2 0   (khoá chưa có -> 0)
print(list(cnt.items()))      # [('t', 1), ('r', 1), ('e', 2)]
print(cnt.most_common())      # đã sắp xếp theo số đếm giảm dần

Counter({'e': 2, 't': 1, 'r': 1})
2 0
[('t', 1), ('r', 1), ('e', 2)]
[('e', 2), ('t', 1), ('r', 1)]


### 3.2. `sorted` với `key` và `lambda`

`sorted(iterable, key=f)` sắp xếp theo giá trị `f(phần_tử)`. `lambda x: x[1]` là hàm vô danh lấy phần tử thứ hai của một cặp. Muốn **giảm dần** theo một số, dùng khoá **âm** (`-x[1]`) hoặc tham số `reverse=True`. `sorted` là sắp xếp **ổn định (stable)**: hai phần tử cùng khoá giữ nguyên thứ tự ban đầu.

In [3]:
pairs = [("t", 1), ("r", 1), ("e", 2)]
print(sorted(pairs, key=lambda p: p[1]))               # tăng dần theo số đếm
print(sorted(pairs, key=lambda p: -p[1]))              # giảm dần: ('e', 2) lên đầu; t, r giữ thứ tự cũ
print(sorted(pairs, key=lambda p: p[1], reverse=True)) # cách khác

[('t', 1), ('r', 1), ('e', 2)]
[('e', 2), ('t', 1), ('r', 1)]
[('e', 2), ('t', 1), ('r', 1)]


### 3.3. Nhân chuỗi và `"".join`

`"e" * 3` cho `"eee"`. `"".join(list_of_str)` ghép các chuỗi con thành một chuỗi (hiệu quả hơn cộng chuỗi trong vòng lặp). Ta sẽ dùng chúng để "in ký tự `c` lặp `f` lần".

In [4]:
print("e" * 3, repr("e" * 0))                       # eee ''
parts = ["e" * 2, "t" * 1, "r" * 1]
print(parts, "->", "".join(parts))                  # ['ee', 't', 'r'] -> eetr

eee ''
['ee', 't', 'r'] -> eetr


### 3.4. `heapq` — hàng đợi ưu tiên (min-heap)

`heapq` trong Python là **min-heap**: `heappop` luôn lấy phần tử **nhỏ nhất**. Muốn lấy phần tử **lớn nhất** trước thì đẩy vào **số đếm âm**: `(-count, ch)`. So sánh cặp (tuple) theo từng phần tử từ trái sang phải: số đếm trước, nếu bằng nhau thì so ký tự. Mỗi `heappush`/`heappop` tốn `O(log k)`.

In [5]:
heap = []
for ch, f in Counter("tree").items():
    heapq.heappush(heap, (-f, ch))
print(heap)                                  # phần tử nhỏ nhất (-2, 'e') ở đầu
while heap:
    f, ch = heapq.heappop(heap)
    print(ch, -f)                            # e 2 / r 1 / t 1  (tie: ký tự nhỏ hơn trước)

[(-2, 'e'), (-1, 't'), (-1, 'r')]
e 2
r 1
t 1


### 3.5. Danh sách các "xô" (bucket) và cái bẫy `[[]] * k`

Bucket sort dùng một mảng, ở vị trí `f` chứa **danh sách các ký tự có tần suất `f`**. Tạo mảng các list rỗng **độc lập** bằng `[[] for _ in range(k)]`. Viết `[[]] * k` tạo `k` **tham chiếu tới cùng một list** — thêm vào xô này sẽ hiện ra ở mọi xô!

In [6]:
bad = [[]] * 3
bad[0].append("x")
print(bad)               # [['x'], ['x'], ['x']]  <- cùng một list

good = [[] for _ in range(3)]
good[0].append("x")
print(good)              # [['x'], [], []]

[['x'], ['x'], ['x']]
[['x'], [], []]


### 3.6. `itertools.groupby` — gom các phần tử giống nhau liền kề

`groupby(out)` trả về từng nhóm ký tự **giống nhau liền kề**: dùng để kiểm tra mỗi ký tự chỉ có **một khối** (hàm kiểm tra ở mục 2). Chú ý: nó **chỉ gom các phần tử liền kề**, nên `"abab"` cho 4 nhóm.

In [7]:
print([(ch, len(list(g))) for ch, g in groupby("aabbbca")])     # [('a', 2), ('b', 3), ('c', 1), ('a', 1)]
print([(ch, len(list(g))) for ch, g in groupby("abab")])        # 4 nhóm -> 'a' bị tách rời

[('a', 2), ('b', 3), ('c', 1), ('a', 1)]
[('a', 1), ('b', 1), ('a', 1), ('b', 1)]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `s = "tree"`:

1. **Đếm:** `t → 1`, `r → 1`, `e → 2`.
2. **Xếp theo số đếm giảm dần:** `e (2)`, rồi `t (1)`, `r (1)` (thứ tự `t`, `r` tuỳ ý).
3. **In mỗi ký tự lặp đúng số lần:** `"ee"` + `"t"` + `"r"` = `"eetr"`.

Với `s = "Aabb"`: `A → 1`, `a → 1`, `b → 2` → `"bb"` + `"A"` + `"a"` = `"bbAa"`.

Hai bước: **đếm**, rồi **xếp theo tần suất**. Bước 2 là chỗ các cách giải khác nhau: sắp xếp thường, heap, hay bucket. Hãy xem trước kết quả của hai bước đó bằng code.

In [8]:
s = "tree"
cnt = Counter(s)
order = sorted(cnt.items(), key=lambda kv: -kv[1])
print("sau khi sắp theo tần suất giảm dần:", order)
out = "".join(ch * f for ch, f in order)
print(out, is_valid_freq_sort(s, out))

sau khi sắp theo tần suất giảm dần: [('e', 2), ('t', 1), ('r', 1)]
eetr True


## 5. Cách 1 — Brute Force / baseline: lấy lần lượt ký tự **nhiều nhất còn lại**

**Ý tưởng.** Mô phỏng đúng cách người ta làm bằng tay:

1. Trong phần chuỗi còn lại, tìm ký tự xuất hiện **nhiều nhất** (dùng `rem.count(c)` cho từng ký tự phân biệt).
2. Ghi `ch * rem.count(ch)` vào kết quả.
3. **Xoá hết** ký tự đó khỏi phần còn lại (`rem.replace(ch, "")`). Lặp đến khi hết.

**Nút thắt.** Mỗi vòng ta đếm lại từ đầu: `k` ký tự phân biệt × `n` ký tự cho mỗi lần `count`, lặp `k` vòng → `O(k² · n)`. Với `k ≤ 62` và `n` lớn thì quá chậm (hàng tỷ thao tác). Đếm đi đếm lại cùng một thứ — chỉ cần đếm **một lần**.

In [9]:
def freq_sort_brute(s):
    rem = s
    out = []
    while rem:
        best = max(set(rem), key=rem.count)       # ký tự có số lần nhiều nhất còn lại
        k = rem.count(best)
        out.append(best * k)
        rem = rem.replace(best, "")
    return "".join(out)


for s, _ in examples:
    r = freq_sort_brute(s)
    assert is_valid_freq_sort(s, r)
    print(f"{s!r:10} -> {r!r}")

'tree'     -> 'eetr'
'cccaaa'   -> 'cccaaa'
'Aabb'     -> 'bbaA'


**Độ phức tạp.** Thời gian `O(k² · n)` (tối đa `62² · n` thao tác cấp C), bộ nhớ `O(n)`. Dùng làm **oracle** cho `n` nhỏ.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** tách bài thành **đếm một lần** + **xếp các (ký tự, số đếm)**:

- Đếm bằng `Counter` trong `O(n)` → chỉ còn tối đa `62` cặp `(ký tự, số đếm)`.
- Xếp `k ≤ 62` cặp theo số đếm giảm dần — gần như miễn phí (`O(k log k)`).
- Dựng chuỗi kết quả: mỗi cặp cho `ch * f`; tổng độ dài `n`.

Ngoài ra, vì tần suất là số nguyên trong `[1, n]`, ta thậm chí **không cần so sánh**: ném mỗi ký tự vào **xô theo tần suất** rồi đọc xô từ `n` về `1` (bucket sort) → `O(n)`.

Hãy xem một con số mới này: số cặp cần xếp nhỏ cỡ nào?

In [10]:
rng = random.Random(0)
big = "".join(rng.choice(string.ascii_letters + string.digits) for _ in range(500_000))
c = Counter(big)
print("độ dài chuỗi:", len(big), "| số ký tự phân biệt:", len(c), "| tần suất lớn nhất:", max(c.values()))

độ dài chuỗi: 500000 | số ký tự phân biệt: 62 | tần suất lớn nhất: 8293


## 7. Các cách tiếp theo

### Cách 2 — `Counter` + sắp xếp các ký tự phân biệt (cách chuẩn)

**Giải thích từng đoạn.**

1. `cnt = Counter(s)` — đếm.
2. `sorted(cnt.items(), key=lambda kv: -kv[1])` — xếp các cặp `(ký tự, số đếm)` theo số đếm **giảm dần**.
3. `"".join(ch * f for ch, f in ...)` — mỗi ký tự lặp `f` lần rồi ghép.

Vì ta xếp **các ký tự phân biệt** (tối đa 62) chứ không xếp cả chuỗi nên thời gian rất nhỏ.

In [11]:
def freq_sort_counter(s):
    cnt = Counter(s)
    return "".join(ch * f for ch, f in sorted(cnt.items(), key=lambda kv: -kv[1]))


for s, _ in examples:
    r = freq_sort_counter(s)
    assert is_valid_freq_sort(s, r)
    print(f"{s!r:10} -> {r!r}")

'tree'     -> 'eetr'
'cccaaa'   -> 'cccaaa'
'Aabb'     -> 'bbAa'


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def frequencySort(self, s: str) -> str:
        cnt = Counter(s)
        return "".join(ch * f for ch, f in sorted(cnt.items(), key=lambda kv: -kv[1]))
```

**Độ phức tạp.** `O(n + k log k)` thời gian (`n` để đếm, `k log k` để sắp xếp, `k ≤ 62`); bộ nhớ `O(n + k)` (chuỗi kết quả + bảng đếm).

### Cách 3 — Dùng **heap** (hàng đợi ưu tiên)

**Ý tưởng khác.** Đẩy mọi cặp `(-số_đếm, ký_tự)` vào min-heap; mỗi lần `heappop` cho ký tự có số đếm **lớn nhất**; ghi `ch * f`. Heap hữu ích khi cần "lấy cái lớn nhất lặp đi lặp lại" mà không cần sắp xếp toàn bộ (ví dụ chỉ lấy top-k).

**Giải thích từng đoạn.** `heap = [(-f, ch) for ch, f in cnt.items()]`; `heapq.heapify(heap)` biến list thành heap trong `O(k)`; vòng `while heap: f, ch = heappop(heap)`; thêm `ch * (-f)`.

In [12]:
def freq_sort_heap(s):
    cnt = Counter(s)
    heap = [(-f, ch) for ch, f in cnt.items()]
    heapq.heapify(heap)                      # O(k)
    out = []
    while heap:
        f, ch = heapq.heappop(heap)          # O(log k): lấy ký tự có số đếm lớn nhất
        out.append(ch * (-f))
    return "".join(out)


for s, _ in examples:
    r = freq_sort_heap(s)
    assert is_valid_freq_sort(s, r)
    print(f"{s!r:10} -> {r!r}")

'tree'     -> 'eert'
'cccaaa'   -> 'aaaccc'
'Aabb'     -> 'bbAa'


**Độ phức tạp.** Thời gian `O(n + k log k)`; bộ nhớ `O(n + k)`. Tương đương Cách 2 ở bài này (vì `k` nhỏ), nhưng là công cụ đúng khi `k` rất lớn hoặc chỉ cần top-`m`.

### Cách 4 — **Bucket sort** theo tần suất (`O(n)`)

**Ý tưởng khác.** Tần suất của mọi ký tự nằm trong `[1, n]`. Tạo `n + 1` xô: xô `f` chứa **các ký tự có đúng `f` lần**. Sau khi đếm, đưa từng ký tự vào xô của nó; rồi đọc xô từ `n` giảm về `1` và ghi `ch * f`. Không cần so sánh, không cần sắp xếp.

**Giải thích từng đoạn.**

1. `cnt = Counter(s)`.
2. `buckets = [[] for _ in range(len(s) + 1)]` (nhớ: **không** dùng `[[]] * (n+1)`).
3. `for ch, f in cnt.items(): buckets[f].append(ch)`.
4. `for f in range(len(s), 0, -1): for ch in buckets[f]: out.append(ch * f)`.

In [13]:
def freq_sort_bucket(s):
    cnt = Counter(s)
    buckets = [[] for _ in range(len(s) + 1)]
    for ch, f in cnt.items():
        buckets[f].append(ch)
    out = []
    for f in range(len(s), 0, -1):           # từ tần suất lớn xuống nhỏ
        for ch in buckets[f]:
            out.append(ch * f)
    return "".join(out)


for s, _ in examples:
    r = freq_sort_bucket(s)
    assert is_valid_freq_sort(s, r)
    print(f"{s!r:10} -> {r!r}")

'tree'     -> 'eetr'
'cccaaa'   -> 'cccaaa'
'Aabb'     -> 'bbAa'


**Độ phức tạp.** Thời gian `O(n + k)` = `O(n)` (đếm `O(n)`, duyệt `n` xô + `k` ký tự); bộ nhớ `O(n)` (mảng xô). Đây là cách có độ phức tạp **tiệm cận tốt nhất** (tuyến tính); thực tế khi `k ≤ 62` thì Cách 2 cũng nhanh tương đương.

### Cách 5 — **Sắp xếp cả chuỗi** với khoá kép `(-tần suất, ký tự)`

**Ý tưởng khác.** Không dựng chuỗi từ các cặp; thay vào đó **sắp xếp trực tiếp mọi ký tự** của `s` theo khoá `(-cnt[c], c)`: ưu tiên tần suất giảm dần; **khi cùng tần suất thì so ký tự** để các ký tự giống nhau **dính liền** (đây là chỗ khoá thứ hai bắt buộc: nếu chỉ dùng `-cnt[c]` thì các ký tự cùng tần suất bị xen kẽ, xem lỗi 2 ở mục 13).

**Giải thích từng đoạn.** `cnt = Counter(s)`; `"".join(sorted(s, key=lambda c: (-cnt[c], c)))`. Chậm hơn Cách 2 vì sắp xếp `n` phần tử (`O(n log n)`), nhưng code gọn một dòng và cho thứ tự **xác định** (tie theo ký tự).

In [14]:
def freq_sort_whole(s):
    cnt = Counter(s)
    return "".join(sorted(s, key=lambda c: (-cnt[c], c)))


for s, _ in examples:
    r = freq_sort_whole(s)
    assert is_valid_freq_sort(s, r)
    print(f"{s!r:10} -> {r!r}")

'tree'     -> 'eert'
'cccaaa'   -> 'aaaccc'
'Aabb'     -> 'bbAa'


**Độ phức tạp.** Thời gian `O(n log n)`; bộ nhớ `O(n)`.

Ghi chú: `Counter.most_common()` đã trả về danh sách `(ký tự, số đếm)` đã sắp xếp giảm dần, nên Cách 2 còn có thể viết `"".join(ch * f for ch, f in Counter(s).most_common())`. Về bản chất vẫn là "đếm + sắp xếp các cặp".

In [15]:
def freq_sort_most_common(s):
    return "".join(ch * f for ch, f in Counter(s).most_common())


for s, _ in examples:
    assert is_valid_freq_sort(s, freq_sort_most_common(s))
print("most_common cũng hợp lệ.")

most_common cũng hợp lệ.


## 8. TRACE từng bước

Ta trace **Cách 4 (bucket)** và **Cách 3 (heap)** trên `s = "tree"` và `s = "cccaaab"` (để có hai ký tự cùng tần suất).

In [16]:
def trace_bucket(s):
    cnt = Counter(s)
    print("1) đếm:", dict(cnt))
    buckets = [[] for _ in range(len(s) + 1)]
    for ch, f in cnt.items():
        buckets[f].append(ch)
        print(f"   ký tự {ch!r} xuất hiện {f} lần -> thêm vào xô {f}")
    print("2) các xô khác rỗng:", {f: b for f, b in enumerate(buckets) if b})
    out = []
    for f in range(len(s), 0, -1):
        for ch in buckets[f]:
            out.append(ch * f)
            print(f"   đọc xô {f}: ghi {ch!r} * {f} = {ch * f!r}, kết quả = {''.join(out)!r}")
    return "".join(out)


r = trace_bucket("tree")
assert is_valid_freq_sort("tree", r)
print()
r = trace_bucket("cccaaab")
assert is_valid_freq_sort("cccaaab", r)

1) đếm: {'t': 1, 'r': 1, 'e': 2}
   ký tự 't' xuất hiện 1 lần -> thêm vào xô 1
   ký tự 'r' xuất hiện 1 lần -> thêm vào xô 1
   ký tự 'e' xuất hiện 2 lần -> thêm vào xô 2
2) các xô khác rỗng: {1: ['t', 'r'], 2: ['e']}
   đọc xô 2: ghi 'e' * 2 = 'ee', kết quả = 'ee'
   đọc xô 1: ghi 't' * 1 = 't', kết quả = 'eet'
   đọc xô 1: ghi 'r' * 1 = 'r', kết quả = 'eetr'

1) đếm: {'c': 3, 'a': 3, 'b': 1}
   ký tự 'c' xuất hiện 3 lần -> thêm vào xô 3
   ký tự 'a' xuất hiện 3 lần -> thêm vào xô 3
   ký tự 'b' xuất hiện 1 lần -> thêm vào xô 1
2) các xô khác rỗng: {1: ['b'], 3: ['c', 'a']}
   đọc xô 3: ghi 'c' * 3 = 'ccc', kết quả = 'ccc'
   đọc xô 3: ghi 'a' * 3 = 'aaa', kết quả = 'cccaaa'
   đọc xô 1: ghi 'b' * 1 = 'b', kết quả = 'cccaaab'


In [17]:
def trace_heap(s):
    cnt = Counter(s)
    heap = [(-f, ch) for ch, f in cnt.items()]
    heapq.heapify(heap)
    print("heap ban đầu (cặp (-đếm, ký tự)):", heap)
    out = []
    step = 0
    while heap:
        step += 1
        f, ch = heapq.heappop(heap)
        out.append(ch * (-f))
        print(f"bước {step}: pop {(f, ch)} -> ghi {ch!r} x {-f}; heap còn {sorted(heap)}; kết quả = {''.join(out)!r}")
    return "".join(out)


r = trace_heap("cccaaab")
assert is_valid_freq_sort("cccaaab", r)

heap ban đầu (cặp (-đếm, ký tự)): [(-3, 'a'), (-3, 'c'), (-1, 'b')]
bước 1: pop (-3, 'a') -> ghi 'a' x 3; heap còn [(-3, 'c'), (-1, 'b')]; kết quả = 'aaa'
bước 2: pop (-3, 'c') -> ghi 'c' x 3; heap còn [(-1, 'b')]; kết quả = 'aaaccc'
bước 3: pop (-1, 'b') -> ghi 'b' x 1; heap còn []; kết quả = 'aaacccb'


Bảng trace Cách 4 trên `"cccaaab"`:

| Bước | Hành động | Trạng thái |
|---|---|---|
| 1 | đếm | `c:3, a:3, b:1` |
| 2 | bỏ vào xô | xô 3 = `[c, a]`, xô 1 = `[b]` |
| 3 | đọc xô 7 → 4 | rỗng, bỏ qua |
| 4 | đọc xô 3 | ghi `ccc`, rồi `aaa` → `"cccaaa"` |
| 5 | đọc xô 2 | rỗng |
| 6 | đọc xô 1 | ghi `b` → `"cccaaab"` |

Điểm cần nhớ: các ký tự trong **cùng một xô** có tần suất bằng nhau nên thứ tự giữa chúng **tuỳ ý** — đúng với "nhiều đáp án đúng".

## 9. Code nộp LeetCode

Bản nộp dùng **Cách 2** (`Counter` + `sorted` các cặp): ngắn gọn, `O(n + k log k)`. Nếu muốn tuyến tính tuyệt đối, dùng Cách 4.

In [18]:
class Solution:
    def frequencySort(self, s: str) -> str:
        cnt = Counter(s)
        return "".join(ch * f for ch, f in sorted(cnt.items(), key=lambda kv: -kv[1]))


sol = Solution()
for s, _ in examples:
    r = sol.frequencySort(s)
    assert is_valid_freq_sort(s, r)
    print(f"{s!r:10} -> {r!r}")

'tree'     -> 'eetr'
'cccaaa'   -> 'cccaaa'
'Aabb'     -> 'bbAa'


## 10. Vì sao thuật toán đúng

Cần chứng minh chuỗi kết quả `out` thoả ba điều kiện của mục 2:

1. **Cùng ký tự, cùng số lần:** mỗi ký tự `ch` được ghi đúng `cnt[ch]` lần (`ch * f`), và mỗi ký tự phân biệt xuất hiện trong danh sách cặp **đúng một lần** → tổng số lần mỗi ký tự trong `out` bằng trong `s`.
2. **Mỗi ký tự liền một khối:** `ch * f` ghi nguyên một khối `f` ký tự giống nhau, và các khối của các ký tự khác nhau được nối tiếp nhau → không ký tự nào bị tách.
3. **Khối sau không dài hơn khối trước:** các cặp được xếp theo `f` giảm dần (hoặc bucket đọc từ lớn xuống nhỏ), nên độ dài các khối tạo thành dãy không tăng.

Vì mọi đáp án hợp lệ có dạng "các khối theo tần suất không tăng", và các ký tự **cùng tần suất** hoán đổi tuỳ ý, nên số đáp án hợp lệ bằng tích các giai thừa số lượng ký tự trong mỗi nhóm cùng tần suất. Ta kiểm chứng điều này **bằng vét cạn** (liệt kê mọi hoán vị của chuỗi nhỏ, dùng một định nghĩa hợp lệ **độc lập** với `is_valid_freq_sort`):

In [19]:
from math import factorial


def valid_by_definition(s, out):
    """Định nghĩa trực tiếp, chậm (O(n^3)), không dùng groupby: độc lập với is_valid_freq_sort."""
    cnt = Counter(s)
    n = len(out)
    if sorted(out) != sorted(s):
        return False
    for i in range(n):
        for j in range(i + 1, n):
            if cnt[out[i]] < cnt[out[j]]:                 # tần suất phải không tăng
                return False
            for k in range(j + 1, n):
                if out[i] == out[k] and out[j] != out[i]: # hai ký tự giống nhau bị tách rời
                    return False
    return True


def count_valid_by_bruteforce(s):
    return sum(1 for p in set(permutations(s)) if valid_by_definition(s, "".join(p)))


def count_valid_by_formula(s):
    groups = Counter(Counter(s).values())        # tần suất -> có bao nhiêu ký tự cùng tần suất
    ans = 1
    for f, m in groups.items():
        ans *= factorial(m)
    return ans


for s in ["tree", "cccaaa", "Aabb", "aabbc", "abcab", "a1a1b", "zzzyyx", "aaaa"]:
    b = count_valid_by_bruteforce(s)
    f = count_valid_by_formula(s)
    assert b == f
    # hai bộ kiểm tra cho cùng kết luận trên mọi hoán vị
    for p in set(permutations(s)):
        assert valid_by_definition(s, "".join(p)) == is_valid_freq_sort(s, "".join(p))
    print(f"{s!r:9}: số đáp án hợp lệ = {b} (công thức tích giai thừa = {f})")

'tree'   : số đáp án hợp lệ = 2 (công thức tích giai thừa = 2)
'cccaaa' : số đáp án hợp lệ = 2 (công thức tích giai thừa = 2)
'Aabb'   : số đáp án hợp lệ = 2 (công thức tích giai thừa = 2)
'aabbc'  : số đáp án hợp lệ = 2 (công thức tích giai thừa = 2)
'abcab'  : số đáp án hợp lệ = 2 (công thức tích giai thừa = 2)
'a1a1b'  : số đáp án hợp lệ = 2 (công thức tích giai thừa = 2)
'zzzyyx' : số đáp án hợp lệ = 1 (công thức tích giai thừa = 1)
'aaaa'   : số đáp án hợp lệ = 1 (công thức tích giai thừa = 1)


## 11. Phân tích Time / Space Complexity

Với `n = len(s)`, `k` = số ký tự phân biệt (`k ≤ 62`):

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Chọn lần lượt max | `O(k² · n)` | `O(n)` | đếm đi đếm lại; chỉ để đối chiếu |
| 2. `Counter` + sort cặp | `O(n + k log k)` | `O(n + k)` | cách chuẩn, ngắn |
| 3. Heap | `O(n + k log k)` | `O(n + k)` | tiện khi cần top-m |
| 4. Bucket sort | `O(n)` | `O(n)` | tuyến tính, không so sánh |
| 5. Sort cả chuỗi | `O(n log n)` | `O(n)` | một dòng, thứ tự xác định |

Chú ý: kết quả có độ dài `n` nên **mọi cách đều `Ω(n)`**; Cách 2–4 là tối ưu về mặt tiệm cận khi `k` coi như hằng số.

## 12. Bộ test

### 12.1. Ví dụ của đề (kiểm tra bằng hàm hợp lệ)

In [20]:
METHODS = {
    "brute": freq_sort_brute,
    "counter": freq_sort_counter,
    "heap": freq_sort_heap,
    "bucket": freq_sort_bucket,
    "whole": freq_sort_whole,
    "most_common": freq_sort_most_common,
    "Solution": Solution().frequencySort,
}


def check_all(s):
    for name, fn in METHODS.items():
        r = fn(s)
        assert is_valid_freq_sort(s, r), (name, s, r)


for s in ["tree", "cccaaa", "Aabb"]:
    check_all(s)
print("3 ví dụ của đề: tất cả các cách đều hợp lệ.")

3 ví dụ của đề: tất cả các cách đều hợp lệ.


### 12.2. Edge case

Một ký tự, toàn ký tự giống nhau, toàn ký tự khác nhau (mọi tần suất 1), chỉ chữ số, chữ hoa/thường xen kẽ, ký tự `'0'`, đủ 62 ký tự, một ký tự rất nhiều và các ký tự khác ít.

In [21]:
edge_cases = [
    "a",
    "aaaaaaaa",
    "abcdefg",
    "1122334455",
    "aAaAaAbB",
    "0",
    "000111",
    string.ascii_letters + string.digits,                         # 62 ký tự, tần suất 1
    (string.ascii_letters + string.digits) * 3,                   # 62 ký tự, tần suất 3
    "x" * 1000 + "y" * 999 + "z",
    "ba" * 50 + "c" * 101,
]
for s in edge_cases:
    check_all(s)
print(f"{len(edge_cases)} edge case: tất cả các cách đều hợp lệ.")

11 edge case: tất cả các cách đều hợp lệ.


### 12.3. Phản ví dụ cho các cách làm sai

Kiểm tra rằng bộ kiểm tra `is_valid_freq_sort` **bắt được** các đáp án sai (nếu không bắt được thì test vô nghĩa).

In [22]:
bad_answers = [
    ("cccaaa", "cacaca"),      # ký tự giống nhau không liền kề
    ("Aabb", "Aabb"),          # tần suất không giảm dần
    ("tree", "tree"),          # 'e' bị tách và tần suất sai
    ("tree", "eertx"),         # thừa ký tự
    ("tree", "eer"),           # thiếu ký tự
    ("aabbb", "aabbb"),        # tăng dần thay vì giảm dần
    ("abab", "abab"),          # xen kẽ
    ("Aabb", "bbaa"),          # coi 'A' = 'a' (thiếu 'A', thừa 'a')
]
for s, wrong in bad_answers:
    assert not is_valid_freq_sort(s, wrong), (s, wrong)
    print(f"{s!r:9} -> {wrong!r:9} bị từ chối (đúng)")
assert not is_valid_freq_sort("tree", ["e", "e", "t", "r"])       # kiểu trả về phải là str
print("Kiểu list cũng bị từ chối.")

'cccaaa'  -> 'cacaca'  bị từ chối (đúng)
'Aabb'    -> 'Aabb'    bị từ chối (đúng)
'tree'    -> 'tree'    bị từ chối (đúng)
'tree'    -> 'eertx'   bị từ chối (đúng)
'tree'    -> 'eer'     bị từ chối (đúng)
'aabbb'   -> 'aabbb'   bị từ chối (đúng)
'abab'    -> 'abab'    bị từ chối (đúng)
'Aabb'    -> 'bbaa'    bị từ chối (đúng)
Kiểu list cũng bị từ chối.


### 12.4. Đối chiếu ngẫu nhiên (có seed) giữa các cách

Với 1000 chuỗi ngẫu nhiên đúng ràng buộc (ký tự trong 62 ký tự; nhiều trường hợp dùng bảng chữ nhỏ để có nhiều tần suất trùng nhau):

- mọi cách cho đáp án **hợp lệ**;
- dãy **độ dài các khối** bằng nhau giữa mọi cách và bằng `sorted(Counter(s).values(), reverse=True)` (vì chỉ thứ tự ký tự cùng tần suất mới được phép khác).

In [23]:
def block_lengths(out):
    return [len(list(g)) for _, g in groupby(out)]


ALPHABET = string.ascii_letters + string.digits
rng = random.Random(451)
for trial in range(1000):
    n = rng.randint(1, 60)
    k = rng.choice([1, 2, 3, 5, 10, 62])
    alpha = rng.sample(ALPHABET, k)
    s = "".join(rng.choice(alpha) for _ in range(n))
    expected_lengths = sorted(Counter(s).values(), reverse=True)
    for name, fn in METHODS.items():
        r = fn(s)
        assert is_valid_freq_sort(s, r), (name, s, r)
        assert block_lengths(r) == expected_lengths, (name, s, r)
print("1000 chuỗi ngẫu nhiên: mọi cách đều hợp lệ và có cùng dãy độ dài khối.")

1000 chuỗi ngẫu nhiên: mọi cách đều hợp lệ và có cùng dãy độ dài khối.


### 12.5. Hiệu năng tại giới hạn `n = 5·10^5`

Đo thời gian trên chuỗi dài nhất của đề (ngẫu nhiên và lệch). Cách 1 (brute) chỉ chạy trên chuỗi ngắn hơn để khỏi quá chậm.

In [24]:
rng = random.Random(5)
uniform = "".join(rng.choice(ALPHABET) for _ in range(500_000))
skew = "".join(rng.choice("aaaaaaaabbbbccd" + ALPHABET) for _ in range(500_000))

for label, s in [("đều", uniform), ("lệch", skew)]:
    print(f"--- chuỗi {label}, n = {len(s)}, k = {len(set(s))}")
    for name in ["counter", "heap", "bucket", "whole", "most_common"]:
        t0 = time.perf_counter()
        r = METHODS[name](s)
        dt = time.perf_counter() - t0
        assert is_valid_freq_sort(s, r)
        print(f"{name:12s} {dt:.3f}s")

small = uniform[:20_000]
t0 = time.perf_counter()
r = freq_sort_brute(small)
print(f"brute (n = 20000): {time.perf_counter() - t0:.3f}s")
assert is_valid_freq_sort(small, r)

--- chuỗi đều, n = 500000, k = 62
counter      0.017s
heap         0.016s


bucket       0.130s


whole        0.219s
most_common  0.019s
--- chuỗi lệch, n = 500000, k = 62
counter      0.015s
heap         0.019s


bucket       0.168s


whole        0.236s
most_common  0.016s
brute (n = 20000): 0.020s


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ sắp xếp theo tần suất, quên gom các ký tự giống nhau

Dùng `sorted(s, key=lambda c: -cnt[c])`: ký tự cùng tần suất **không được gom** (sắp xếp ổn định giữ nguyên thứ tự ban đầu của chúng, nên bị xen kẽ).

In [25]:
def wrong_only_freq(s):
    cnt = Counter(s)
    return "".join(sorted(s, key=lambda c: -cnt[c]))      # SAI: cùng tần suất -> giữ thứ tự cũ, xen kẽ


s = "cacaca"
r = wrong_only_freq(s)
print(repr(r), "hợp lệ:", is_valid_freq_sort(s, r))
assert not is_valid_freq_sort(s, r)
print(repr(freq_sort_whole(s)), "hợp lệ:", is_valid_freq_sort(s, freq_sort_whole(s)))   # khoá kép (-tần suất, ký tự)

'cacaca' hợp lệ: False
'aaaccc' hợp lệ: True


**Vì sao sai:** `"cacaca"` có `c:3, a:3` — cùng khoá `-3` nên `sorted` ổn định giữ nguyên `"cacaca"`. **Đúng:** thêm khoá phụ `c` (Cách 5) hoặc dựng bằng `ch * f` (Cách 2–4).

### Lỗi 2: Coi chữ hoa và chữ thường là một

In [26]:
def wrong_case_insensitive(s):
    cnt = Counter(s.lower())                           # SAI: 'A' và 'a' bị gộp
    return "".join(ch * f for ch, f in sorted(cnt.items(), key=lambda kv: -kv[1]))


s = "Aabb"
r = wrong_case_insensitive(s)
print(repr(r), "hợp lệ:", is_valid_freq_sort(s, r))
assert not is_valid_freq_sort(s, r)

'aabb' hợp lệ: False


**Vì sao sai:** đề nói `'A'` và `'a'` là **hai ký tự khác nhau**; gộp lại sẽ làm mất chữ `A` (đầu ra toàn chữ thường). **Đúng:** đếm trên chuỗi nguyên bản `Counter(s)`.

### Lỗi 3: Trả về `list` thay vì `str`

In [27]:
def wrong_return_list(s):
    cnt = Counter(s)
    return [ch * f for ch, f in sorted(cnt.items(), key=lambda kv: -kv[1])]    # SAI: quên "".join


r = wrong_return_list("tree")
print(r, type(r).__name__, "hợp lệ:", is_valid_freq_sort("tree", r))
assert not is_valid_freq_sort("tree", r)

['ee', 't', 'r'] list hợp lệ: False


**Vì sao sai:** đề yêu cầu trả về **chuỗi**. `["ee", "t", "r"]` là list các chuỗi con. **Đúng:** `"".join(...)`.

### Lỗi 4: Sắp xếp tăng dần thay vì giảm dần

In [28]:
def wrong_ascending(s):
    cnt = Counter(s)
    return "".join(ch * f for ch, f in sorted(cnt.items(), key=lambda kv: kv[1]))   # SAI: thiếu dấu trừ


s = "tree"
r = wrong_ascending(s)
print(repr(r), "hợp lệ:", is_valid_freq_sort(s, r))
assert not is_valid_freq_sort(s, r)

'tree' hợp lệ: False


**Vì sao sai:** thiếu dấu `-` (hoặc `reverse=True`) làm ký tự ít xuất hiện đứng trước. **Đúng:** `key=lambda kv: -kv[1]`.

### Lỗi 5: Dùng `[[]] * k` để tạo xô

In [29]:
def wrong_shared_buckets(s):
    cnt = Counter(s)
    buckets = [[]] * (len(s) + 1)                     # SAI: mọi xô là CÙNG một list
    for ch, f in cnt.items():
        buckets[f].append(ch)
    out = []
    for f in range(len(s), 0, -1):
        for ch in buckets[f]:
            out.append(ch * f)
    return "".join(out)


s = "tree"
r = wrong_shared_buckets(s)
print(repr(r), "hợp lệ:", is_valid_freq_sort(s, r))
assert not is_valid_freq_sort(s, r)

'ttttrrrreeeetttrrreeettrreetre' hợp lệ: False


**Vì sao sai:** mọi `ch` được thêm vào **cùng một list** nên xô nào cũng chứa cả `e`, `t`, `r`; khi đọc, ký tự bị ghi nhiều lần. **Đúng:** `[[] for _ in range(n + 1)]`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Sắp xếp theo tần suất"** → luôn nghĩ **đếm trước**, rồi mới sắp xếp các cặp `(giá trị, số đếm)`.
- Số đếm là số nguyên trong `[1, n]` → có thể dùng **bucket sort** (chỉ số xô = số đếm) để được `O(n)`.
- **Top-k phần tử tần suất cao** → heap (kích thước `k`) hoặc `most_common(k)`; bucket cũng được (bài 347).
- Muốn lấy **lớn nhất** từ `heapq` (min-heap) → đẩy giá trị **âm**.
- Sắp xếp theo hai tiêu chí → dùng **tuple làm khoá**: `key=lambda c: (-cnt[c], c)`.
- **Nhiều đáp án đúng** → viết hàm kiểm tra tính chất thay vì so với chuỗi cứng; kiểm chứng hàm kiểm tra bằng vét cạn trên chuỗi nhỏ (mục 10).
- Nối chuỗi: dùng `"".join(list)` thay vì `+=` trong vòng lặp.

## 15. Biến thể của bài

1. **Sắp xếp tần suất tăng dần** (hoặc giảm dần, cùng tần suất thì theo bảng chữ cái — kết quả **xác định**).
2. **Top K Frequent Elements** (347): trả về `k` giá trị xuất hiện nhiều nhất.
3. **Sort Array by Increasing Frequency** (1636): sắp xếp mảng số; cùng tần suất thì **giá trị giảm dần**.
4. **Reorganize String** (767): sắp xếp lại để **không có hai ký tự giống nhau kề nhau** — dùng max-heap lấy ký tự nhiều nhất còn lại.
5. **First Unique Character in a String** (387): chỉ cần `Counter`.
6. **Sắp xếp các từ** theo tần suất thay vì ký tự (xem bài tập 5).

Ví dụ biến thể 3 (cùng tần suất → giá trị giảm dần), kết quả xác định nên có thể dùng `assert` trực tiếp:

In [30]:
def frequency_sort_array(nums):
    cnt = Counter(nums)
    return sorted(nums, key=lambda x: (cnt[x], -x))     # tần suất tăng dần, cùng tần suất thì x giảm dần


print(frequency_sort_array([1, 1, 2, 2, 2, 3]))        # [3, 1, 1, 2, 2, 2]
print(frequency_sort_array([2, 3, 1, 3, 2]))           # [1, 3, 3, 2, 2]
assert frequency_sort_array([1, 1, 2, 2, 2, 3]) == [3, 1, 1, 2, 2, 2]
assert frequency_sort_array([2, 3, 1, 3, 2]) == [1, 3, 3, 2, 2]
assert frequency_sort_array([-1, 1, -6, 4, 5, -6, 1, 4, 1]) == [5, -1, 4, 4, -6, -6, 1, 1, 1]

[3, 1, 1, 2, 2, 2]
[1, 3, 3, 2, 2]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Chọn lần lượt max | `count` + `replace` lặp lại | `O(k² · n)` | `O(n)` | chỉ để đối chiếu, `n` nhỏ |
| 2. `Counter` + sort cặp | đếm, sắp xếp ≤ 62 cặp | `O(n + k log k)` | `O(n + k)` | **cách nộp bài** (ngắn, nhanh) |
| 3. Heap | max-heap bằng số âm | `O(n + k log k)` | `O(n + k)` | cần top-m, `k` lớn |
| 4. Bucket sort | xô theo tần suất | `O(n)` | `O(n)` | muốn tuyến tính tuyệt đối |
| 5. Sort cả chuỗi | khoá `(-tần suất, ký tự)` | `O(n log n)` | `O(n)` | code một dòng, thứ tự xác định |

Quy tắc chọn: nộp bài → Cách 2; hỏi "có cách `O(n)` không?" → Cách 4; hỏi về top-k → Cách 3.

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `Counter("abbccc").most_common()`, `is_valid_freq_sort("tree", "reet")`, `is_valid_freq_sort("tree", "eert")`, `is_valid_freq_sort("tree", "ertx")`, `freq_sort_whole("abbccc")`.

**Bài 2 (tính tay).** (a) Với `s = "abbcccdddd1"` liệt kê tần suất từng ký tự và cho biết **có bao nhiêu** đáp án hợp lệ. (b) Với `s = "aabbcc"` có bao nhiêu đáp án hợp lệ? Viết ra 3 đáp án.

**Bài 3 (điền TODO).** Hoàn thành `freq_sort_todo(s)` bằng bucket sort.

In [31]:
def freq_sort_todo(s):
    # TODO 1: cnt = Counter(s)
    # TODO 2: buckets = [[] for _ in range(len(s) + 1)]
    # TODO 3: với mỗi (ch, f) trong cnt.items(): buckets[f].append(ch)
    # TODO 4: out = []; với f từ len(s) xuống 1: với mỗi ch trong buckets[f]: out.append(ch * f)
    # TODO 5: return "".join(out)
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `freq_sort_deterministic(s)` cho kết quả **xác định**: tần suất giảm dần, cùng tần suất thì **ký tự theo bảng mã tăng dần** (không dùng `sorted(s, ...)` trên cả chuỗi; dùng `sorted` trên các cặp).

**Bài 5 (biến thể / mini challenge).** (a) Viết `freq_sort_ascending(s)`: tần suất **tăng dần**, cùng tần suất thì ký tự tăng dần; kiểm tra bằng `block_lengths` tăng dần. (b) Viết `sort_words_by_frequency(words)` trả về **list từ** sắp theo tần suất giảm dần, mỗi từ xuất hiện **liền thành một khối** (ví dụ `["b","a","b"] -> ["b","b","a"]` hoặc `["a"...]` tuỳ tie); viết hàm kiểm tra riêng và đối chiếu ngẫu nhiên.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[('c', 3), ('b', 2), ('a', 1)]`; `"reet"` **không** hợp lệ (các khối có độ dài 1, 2, 1 — khối `r` dài 1 đứng trước khối `ee` dài 2 nên tần suất không giảm dần); `"eert"` hợp lệ; `"ertx"` không hợp lệ (thừa `x`); `freq_sort_whole("abbccc")` = `"cccbba"`. Bài 3: bám sát mã Cách 4 ở mục 7.
- **Mức 2 (vừa):** Bài 2: (a) `a,1`, `b,2`, `c,3`, `d,4`, `1,1`: chỉ nhóm tần suất `1` có 2 ký tự (`a` và `1`) → `2!` = 2 đáp án; (b) cả ba ký tự cùng tần suất 2 → `3! = 6` đáp án (`aabbcc`, `aaccbb`, `bbaacc`, ...). Bài 4: `sorted(cnt.items(), key=lambda kv: (-kv[1], kv[0]))`.
- **Mức 3 (sâu):** Bài 5(a): khoá `(f, ch)` rồi ghép `ch * f`; kiểm tra bằng `groupby` rằng độ dài khối không giảm. 5(b): đếm bằng `Counter(words)`, dựng list bằng `[w] * f`; hàm kiểm tra: dùng `groupby` trên list các từ, mỗi từ một khối, độ dài khối không tăng, `Counter` bằng nhau.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [32]:
# Bài 1
assert Counter("abbccc").most_common() == [("c", 3), ("b", 2), ("a", 1)]
assert not is_valid_freq_sort("tree", "reet")
assert is_valid_freq_sort("tree", "eert")
assert not is_valid_freq_sort("tree", "ertx")
assert freq_sort_whole("abbccc") == "cccbba"

# Bài 2
assert count_valid_by_formula("abbcccdddd1") == 2
assert count_valid_by_formula("aabbcc") == 6
for ans in ["aabbcc", "aaccbb", "bbaacc"]:
    assert is_valid_freq_sort("aabbcc", ans)


# Bài 3
def freq_sort_todo_solution(s):
    cnt = Counter(s)
    buckets = [[] for _ in range(len(s) + 1)]
    for ch, f in cnt.items():
        buckets[f].append(ch)
    out = []
    for f in range(len(s), 0, -1):
        for ch in buckets[f]:
            out.append(ch * f)
    return "".join(out)


# Bài 4
def freq_sort_deterministic(s):
    cnt = Counter(s)
    return "".join(ch * f for ch, f in sorted(cnt.items(), key=lambda kv: (-kv[1], kv[0])))


# Bài 5a
def freq_sort_ascending(s):
    cnt = Counter(s)
    return "".join(ch * f for ch, f in sorted(cnt.items(), key=lambda kv: (kv[1], kv[0])))


# Bài 5b
def sort_words_by_frequency(words):
    cnt = Counter(words)
    out = []
    for w, f in sorted(cnt.items(), key=lambda kv: -kv[1]):
        out.extend([w] * f)
    return out


def is_valid_word_sort(words, out):
    if Counter(out) != Counter(words):
        return False
    seen, lengths = set(), []
    for w, g in groupby(out):
        if w in seen:
            return False
        seen.add(w)
        lengths.append(len(list(g)))
    return all(a >= b for a, b in zip(lengths, lengths[1:]))


assert freq_sort_deterministic("tree") == "eert" and freq_sort_deterministic("cccaaa") == "aaaccc"
assert freq_sort_deterministic("Aabb") == "bbAa"
assert freq_sort_ascending("tree") == "rtee"

rng = random.Random(2024)
for _ in range(500):
    n = rng.randint(1, 40)
    alpha = rng.sample(ALPHABET, rng.choice([1, 2, 4, 10]))
    s = "".join(rng.choice(alpha) for _ in range(n))
    assert is_valid_freq_sort(s, freq_sort_todo_solution(s))
    d = freq_sort_deterministic(s)
    assert is_valid_freq_sort(s, d) and d == freq_sort_deterministic(s)
    a = freq_sort_ascending(s)
    assert Counter(a) == Counter(s)
    lens = block_lengths(a)
    assert lens == sorted(lens) and len(set(a)) == len(lens)
    words = [rng.choice(["x", "yy", "z", "w"]) for _ in range(rng.randint(0, 25))]
    assert is_valid_word_sort(words, sort_words_by_frequency(words))
print("Tất cả đáp án bài tập đều đúng.")

Tất cả đáp án bài tập đều đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (sắp xếp theo tần suất: đếm → sắp xếp cặp → dựng chuỗi)

```python
class Solution:
    def frequencySort(self, s: str) -> str:
        cnt = Counter(s)                                           # 1) đếm một lần
        pairs = sorted(cnt.items(), key=lambda kv: -kv[1])         # 2) xếp cặp theo số đếm giảm dần
        return "".join(ch * f for ch, f in pairs)                  # 3) ch lặp f lần, ghép lại
```

Biến thể tuyến tính (bucket): `buckets = [[] for _ in range(n + 1)]`; `buckets[f].append(ch)`; đọc `f` từ `n` xuống `1`.

**Checklist**

- [ ] Đếm trên chuỗi **nguyên bản** (phân biệt chữ hoa/thường)?
- [ ] Số đếm xếp **giảm dần** (khoá âm hoặc `reverse=True`)?
- [ ] Các ký tự giống nhau **liền một khối** (dựng bằng `ch * f`, hoặc khoá phụ `c` nếu sort cả chuỗi)?
- [ ] Trả về `str` (đã `"".join`), không phải `list`?
- [ ] Nếu dùng bucket: `[[] for _ in range(n + 1)]` chứ không phải `[[]] * (n + 1)`?
- [ ] Test bằng hàm kiểm tra tính hợp lệ (không so chuỗi cứng) vì có nhiều đáp án đúng?
- [ ] Đã thử `"a"`, toàn ký tự khác nhau, toàn một ký tự, `"Aabb"`, `n = 5·10^5`?
- [ ] Skeleton đúng `class Solution` / `def frequencySort(self, s: str) -> str`?